## what to do
 - load data
 - scale (save as json)
 - build mode
 - eval
 - quant
 - export to h file and tflite

In [126]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [127]:
labels_names = pd.read_csv("dataset_uci/activity_labels.txt", header=None, delimiter=" ", names=["label", "activity"])
labels_names

,label,activity
0,1,WALKING
1,2,WALKING_UPSTAIRS
2,3,WALKING_DOWNSTAIRS
3,4,SITTING
4,5,STANDING
5,6,LAYING


In [128]:
x_train = pd.read_csv("dataset_uci/final_X_train.txt", header=None)
y_train = pd.read_csv("dataset_uci/final_y_train.txt", header=None)

x_test = pd.read_csv("dataset_uci/final_X_test.txt", header=None)
y_test = pd.read_csv("dataset_uci/final_y_test.txt", header=None)

# print range ot y values
print("y_train range:", y_train[0].min(), "to", y_train[0].max())
print("y_test range:", y_test[0].min(), "to", y_test[0].max())

# make it from 0 to 5
y_train -= 1
y_test -= 1

print("y_train range:", y_train[0].min(), "to", y_train[0].max())
print("y_test range:", y_test[0].min(), "to", y_test[0].max())



y_train range: 1 to 6
y_test range: 1 to 6
y_train range: 0 to 5
y_test range: 0 to 5


In [129]:
# shapes

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("--------------------------------")
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

x_train shape: (4252, 561)
y_train shape: (4252, 1)
--------------------------------
x_test shape: (1492, 561)
y_test shape: (1492, 1)


In [130]:
# data types
x_train.dtypes

0      float64
1      float64
2      float64
3      float64
4      float64
        ...   
556    float64
557    float64
558    float64
559    float64
560    float64
Length: 561, dtype: object

In [131]:
# missing values
x_train.isnull().sum()

0      0
1      0
2      0
3      0
4      0
      ..
556    0
557    0
558    0
559    0
560    0
Length: 561, dtype: int64

In [132]:
# class distribution for training set
print("Class distribution for training set:")
print(y_train.value_counts())
print("--------------------------------")
# class distribution for test set
print("Class distribution for test set:")
print(y_test.value_counts())


Class distribution for training set:
0
3    834
4    775
0    769
2    691
1    629
5    554
Name: count, dtype: int64
--------------------------------
Class distribution for test set:
0
3    289
4    254
0    243
2    239
5    238
1    229
Name: count, dtype: int64


## Data Inspection

In [133]:
# check ranges of the train data

print("Train data ranges:")
print(x_train.min().min(), "to", x_train.max().max())

Train data ranges:
-3.1233 to 7.0658


In [134]:
x_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4252 entries, 0 to 4251
Columns: 561 entries, 0 to 560
dtypes: float64(561)
memory usage: 18.2 MB


In [135]:
# data types are all float64
x_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4252 entries, 0 to 4251
Columns: 561 entries, 0 to 560
dtypes: float64(561)
memory usage: 18.2 MB


In [136]:
# check for duplicates
x_train.duplicated().sum()

# drop duplicates if any
x_train = x_train.drop_duplicates()
y_train = y_train.loc[x_train.index]


## Preprocessing
- Separate features and labels -> already done
- Encode labels into integers -> already done 

In [137]:
#Apply feature scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("Train data ranges after scaling:")
print(x_train_scaled.min().min(), "to", x_train_scaled.max().max())

Train data ranges after scaling:
-16.854827746384903 to 55.564232635091834


In [138]:
import json

# feature order as json file
feature_order = x_train.columns.tolist()

# scaler parameters
scaler_params = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
}
# label mapping
label_mapping = dict(zip(labels_names["label"], labels_names["activity"]))

# Export all preprocessing metadata to preprocessing.json
preprocessing_metadata = {
    "feature_order": feature_order,
    "scaler_params": scaler_params,
    "label_mapping": label_mapping,
}
with open("preprocessing.json", "w") as f:
    json.dump(preprocessing_metadata, f, indent=4)

## Model Design

In [139]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

In [140]:
INPUT_SHAPE = x_train_scaled.shape[1]
NUM_CLASSES = y_train[0].nunique()

print("Input shape:", INPUT_SHAPE)
print("Number of classes:", NUM_CLASSES)

# print uniqe values in y_train
print("Unique labels in y_train:", y_train[0].unique())

Input shape: 561
Number of classes: 6
Unique labels in y_train: [0 1 2 3 4 5]


In [ ]:
# hyperparameters as a dict
hyperparameters = {
    "learning_rate": [0.0001, 0.001, 0.01],
    "batch_size": [16, 32, 64],
    "dense_units_1": [128, 256, 512],
    "dense_units_2": [128, 256, 512],
    "dropout_rate_1": [0.2, 0.3, 0.5],
    "dropout_rate_2": [0.2, 0.3, 0.5],
}


In [142]:
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(INPUT_SHAPE,)))
    model.add(layers.Dense(hp.Choice("dense_units_1", hyperparameters["dense_units_1"]), activation="relu"))
    model.add(layers.Dropout(hp.Choice("dropout_rate_1", hyperparameters["dropout_rate_1"])))
    model.add(layers.Dense(hp.Choice("dense_units_2",hyperparameters["dense_units_2"]), activation="relu"))
    model.add(layers.Dropout(hp.Choice("dropout_rate_2", hyperparameters["dropout_rate_2"])))
    model.add(layers.Dense(NUM_CLASSES, activation="softmax"))

    optimizer = Adam(learning_rate=hp.Choice("learning_rate", hyperparameters["learning_rate"]))
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [143]:
from kerastuner import RandomSearch

turner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=3,
    directory='my_dir',
    project_name='human_activity_recognition'
)


Reloading Tuner from my_dir/human_activity_recognition/tuner0.json


In [144]:
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping

checkpoint = ModelCheckpoint(
    filepath='best_model.h5',
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True,
    mode='max'
)

earlystop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    mode='max',
    verbose=1
)

In [145]:
y_train = y_train.values.flatten()
y_test = y_test.values.flatten()

turner.search(
    x_train_scaled, y_train,
    validation_data=(x_test_scaled, y_test),
    epochs= 10,
    callbacks=[checkpoint, earlystop],
)


Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
64                |256               |dense_units_1
0.2               |0.5               |dropout_rate_1
128               |32                |dense_units_2
0.3               |0.3               |dropout_rate_2
0.1               |0.1               |learning_rate

Epoch 1/10
126/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2160 - loss: 19.7832
Epoch 1: val_accuracy improved from None to 0.16287, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1944 - loss: 6.7071 - val_accuracy: 0.1629 - val_loss: 1.8041
Epoch 2/10
118/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1807 - loss: 1.7935
Epoch 2: val_accuracy improved from 0.16287 to 0.19370, saving model to best_model.h5



Epoch 2: finished saving model to best_model.h5
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1768 - loss: 1.7962 - val_accuracy: 0.1937 - val_loss: 1.7923
Epoch 3/10
125/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1921 - loss: 1.7893
Epoch 3: val_accuracy did not improve from 0.19370
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1920 - loss: 1.7913 - val_accuracy: 0.1937 - val_loss: 1.8009
Epoch 4/10
106/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1974 - loss: 1.7964
Epoch 4: val_accuracy did not improve from 0.19370
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1932 - loss: 1.8134 - val_accuracy: 0.1702 - val_loss: 1.7932
Epoch 5/10
111/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1957 - loss: 1.8110
Epoch 5: val_accuracy did not improve from 0.19370
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1965 - loss: 1.8125 - val_accuracy: 0.1937 - val_loss: 1.7942
Epoch 6/10
128/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1970 -


Epoch 1: finished saving model to best_model.h5
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1998 - loss: 8.2788 - val_accuracy: 0.1937 - val_loss: 1.8118
Epoch 2/10
124/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1975 - loss: 1.9944
Epoch 2: val_accuracy did not improve from 0.19370
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1918 - loss: 1.8452 - val_accuracy: 0.1702 - val_loss: 1.8127
Epoch 3/10
120/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1929 - loss: 1.9992
Epoch 3: val_accuracy improved from 0.19370 to 0.19437, saving model to best_model.h5



Epoch 3: finished saving model to best_model.h5
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1868 - loss: 2.0182 - val_accuracy: 0.1944 - val_loss: 1.8121
Epoch 4/10
126/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1893 - loss: 1.8042
Epoch 4: val_accuracy did not improve from 0.19437
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1946 - loss: 1.8194 - val_accuracy: 0.1602 - val_loss: 1.7978
Epoch 5/10
125/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1930 - loss: 1.7947
Epoch 5: val_accuracy did not improve from 0.19437
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1839 - loss: 1.7943 - val_accuracy: 0.1702 - val_loss: 1.7988
Epoch 6/10
 83/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1910 - loss: 1.7932

KeyboardInterrupt: 